In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import embeddings as emb
import inference as inf
import boto3
from pathlib import Path

## Load vector stores from S3

In [3]:
# Load recursive split embeddings for model "intfloat/multilingual-e5-large-instruct"
e5_large_recursive = emb.load_embedding_vector_store(model="intfloat/multilingual-e5-large-instruct", splitter_type="recursive")

Loading locally available vector store.


In [4]:
# Load unstructured (basic) split embeddings for model "intfloat/multilingual-e5-large-instruct"
e5_large_unstruct_basic = emb.load_embedding_vector_store(model="intfloat/multilingual-e5-large-instruct", splitter_type="unstruct_basic")

Loading locally available vector store.


In [5]:
# Load unstructured (basic) split embeddings for model "intfloat/multilingual-e5-large-instruct"
e5_large_unstruct_by_title = emb.load_embedding_vector_store(model="intfloat/multilingual-e5-large-instruct", splitter_type="unstruct_by_title")

Loading locally available vector store.


In [6]:
# Load recursive split embeddings for model "intfloat/e5-small-v2"
e5_small_recursive = emb.load_embedding_vector_store(model="intfloat/e5-small-v2", splitter_type="recursive")

Loading locally available vector store.


In [7]:
# Load unstructured (basic) split embeddings for model "intfloat/e5-small-v2"
e5_small_unstruct_basic = emb.load_embedding_vector_store(model="intfloat/e5-small-v2", splitter_type="unstruct_basic")

Loading locally available vector store.


In [8]:
# Load unstructured (basic) split embeddings for model "intfloat/e5-small-v2"
e5_small_unstruct_by_title = emb.load_embedding_vector_store(model="intfloat/e5-small-v2", splitter_type="unstruct_by_title")

Loading locally available vector store.


## Get test queries from S3

In [9]:
# Get user queries from S3
import pandas as pd 

base_dir = Path.cwd().parent 
local_file_dir = base_dir / "temp" 
emb.download_from_s3(s3_path="context-test-questions.csv", output_path=local_file_dir)
queries = pd.read_csv(f"{local_file_dir}/context-test-questions.csv")

Saved file from s3://local-gov-ai-llm-benchmarking/context-test-questions.csv at /home/jovyan/llm-benchmarking/temp.


## Get context for all embedding types

In [10]:
# Define dictionary for embedding models 
embeddings = {
    "e5_large_recursive" : e5_large_recursive, 
    "e5_large_unstruct_basic" : e5_large_unstruct_basic,
    "e5_large_unstruct_by_title" : e5_large_unstruct_by_title,  
    "e5_small_recursive" : e5_small_recursive,  
    "e5_small_unstruct_basic" : e5_small_unstruct_basic,
    "e5_small_unstruct_by_title" : e5_small_unstruct_by_title
}


In [11]:
# Get context for all queries and embeddings 
all_iterations = pd.DataFrame()

for emb_name, emb_model in embeddings.items():
    df = inf.get_test_queries_with_context(queries, emb_model, 6)
    df['emb_type'] = emb_name
    all_iterations = pd.concat([all_iterations, df.copy()], ignore_index=True)

In [12]:
# Save context
all_iterations.to_csv(f"{local_file_dir}/context-test-questions-output.csv", index=False)